<a href="https://colab.research.google.com/github/vitorpdim/agenteLLM/blob/main/LM_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings
import logging

warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("litellm").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)

import dspy
from dspy.teleprompt import BootstrapFewShot

import subprocess, threading, time, requests

def configurar_ambiente():
    pacotes = ['dspy-ai', 'duckduckgo-search']
    print(f"[1/4] Instalando pacotes Python: {', '.join(pacotes)}...")
    subprocess.run(['pip', 'install', '-q'] + pacotes, check=True)

    print("[2/4] Instalando dependência do sistema (zstd)...")
    subprocess.run(['sudo', 'apt-get', 'install', '-y', 'zstd'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print("[3/4] Instalando ollama...")
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    threading.Thread(
        target=lambda: subprocess.Popen(['ollama', 'serve'],
                                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL),
        daemon=True
    ).start()

    print("      Aguardando servidor subir", end="", flush=True)
    for i in range(20):
        try:
            if requests.get('http://localhost:11434', timeout=2).status_code == 200:
                print(f" — ok ({(i+1)*2}s)")
                break
        except requests.ConnectionError:
            print(".", end="", flush=True)
            time.sleep(2)
    else:
        raise RuntimeError("Ollama não respondeu a tempo!")

    print("[4/4] Baixando gemma3:1b (pode demorar na primeira vez)...")
    subprocess.run(['ollama', 'pull', 'gemma3:1b'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("\nAmbiente pronto!\n")

configurar_ambiente()

[1/4] Instalando pacotes Python: dspy-ai, duckduckgo-search...
[2/4] Instalando dependência do sistema (zstd)...
[3/4] Instalando ollama...
      Aguardando servidor subir. — ok (4s)
[4/4] Baixando gemma3:1b (pode demorar na primeira vez)...

Ambiente pronto!



In [2]:
import sqlite3

def configurar_banco_dados(caminho='empresa.db'):
    with sqlite3.connect(caminho) as conn:
        c = conn.cursor()
        c.executescript('''
            CREATE TABLE IF NOT EXISTS departamentos (
                id INTEGER PRIMARY KEY, nome TEXT, orcamento INTEGER,
                gerente_id INTEGER, localizacao TEXT
            );
            CREATE TABLE IF NOT EXISTS funcionarios (
                id INTEGER PRIMARY KEY, nome TEXT, departamento_id INTEGER,
                cargo TEXT, salario INTEGER, data_contratacao TEXT,
                email TEXT, cidade TEXT, ativo INTEGER DEFAULT 1
            );
            CREATE TABLE IF NOT EXISTS projetos (
                id INTEGER PRIMARY KEY, nome TEXT, departamento_id INTEGER,
                orcamento INTEGER, status TEXT,
                data_inicio TEXT, data_fim TEXT, prioridade TEXT
            );
            CREATE TABLE IF NOT EXISTS alocacoes (
                id INTEGER PRIMARY KEY, funcionario_id INTEGER,
                projeto_id INTEGER, horas_semanais INTEGER, papel TEXT
            );
            CREATE TABLE IF NOT EXISTS avaliacoes (
                id INTEGER PRIMARY KEY, funcionario_id INTEGER,
                ano INTEGER, nota REAL, comentario TEXT
            );
        ''')

        departamentos = [
            (1, 'Engenharia',       800000,  1, 'São Paulo'),
            (2, 'Vendas',           400000,  6, 'Rio de Janeiro'),
            (3, 'Recursos Humanos', 200000, 11, 'São Paulo'),
            (4, 'Marketing',        300000, 15, 'Belo Horizonte'),
            (5, 'Financeiro',       250000, 19, 'São Paulo'),
            (6, 'Suporte Técnico',  180000, 23, 'Curitiba'),
        ]

        funcionarios = [
            # Engenharia
            (1,  'Alice Silva',      1, 'Gerente de Engenharia',  120000, '2020-01-15', 'alice@emp.com',     'São Paulo',      1),
            (2,  'Bruno Santos',     1, 'Engenheiro Sênior',       95000, '2021-03-10', 'bruno@emp.com',     'São Paulo',      1),
            (3,  'Carla Oliveira',   1, 'Engenheira Plena',        80000, '2022-07-20', 'carla@emp.com',     'Campinas',       1),
            (4,  'Daniel Ferreira',  1, 'Engenheiro Júnior',       60000, '2023-01-05', 'daniel@emp.com',    'São Paulo',      1),
            (5,  'Elena Costa',      1, 'DevOps Engineer',         88000, '2021-09-12', 'elena@emp.com',     'São Paulo',      1),
            # Vendas
            (6,  'Felipe Rocha',     2, 'Gerente de Vendas',      110000, '2019-05-22', 'felipe@emp.com',    'Rio de Janeiro', 1),
            (7,  'Gabriela Lima',    2, 'Vendedora Sênior',        75000, '2020-11-30', 'gabriela@emp.com',  'Rio de Janeiro', 1),
            (8,  'Henrique Moura',   2, 'Vendedor Pleno',          65000, '2022-02-14', 'henrique@emp.com',  'Niterói',        1),
            (9,  'Isabela Nunes',    2, 'Vendedora Júnior',        50000, '2023-06-01', 'isabela@emp.com',   'Rio de Janeiro', 1),
            (10, 'João Pereira',     2, 'Analista de CRM',         58000, '2022-08-17', 'joao@emp.com',      'Rio de Janeiro', 1),
            # RH
            (11, 'Karina Mendes',    3, 'Gerente de RH',          105000, '2018-04-03', 'karina@emp.com',    'São Paulo',      1),
            (12, 'Lucas Barbosa',    3, 'Analista de RH Sênior',   72000, '2020-10-25', 'lucas@emp.com',     'São Paulo',      1),
            (13, 'Marina Cardoso',   3, 'Analista de RH Pleno',    62000, '2021-12-08', 'marina@emp.com',    'São Paulo',      1),
            (14, 'Nelson Souza',     3, 'Recrutador',              55000, '2023-03-20', 'nelson@emp.com',    'São Paulo',      1),
            # Marketing
            (15, 'Olivia Martins',   4, 'Gerente de Marketing',   108000, '2019-07-11', 'olivia@emp.com',    'Belo Horizonte', 1),
            (16, 'Paulo Teixeira',   4, 'Designer Sênior',         78000, '2020-09-05', 'paulo@emp.com',     'Belo Horizonte', 1),
            (17, 'Queila Rodrigues', 4, 'Analista de Conteúdo',    60000, '2022-04-18', 'queila@emp.com',    'Belo Horizonte', 1),
            (18, 'Rafael Alves',     4, 'Especialista em SEO',     67000, '2021-06-30', 'rafael@emp.com',    'São Paulo',      1),
            # Financeiro
            (19, 'Sabrina Castro',   5, 'Gerente Financeiro',     115000, '2017-11-20', 'sabrina@emp.com',   'São Paulo',      1),
            (20, 'Thiago Pinto',     5, 'Contador Sênior',         82000, '2019-03-14', 'thiago@emp.com',    'São Paulo',      1),
            (21, 'Ursula Freitas',   5, 'Analista Financeiro',     68000, '2021-08-22', 'ursula@emp.com',    'São Paulo',      1),
            (22, 'Victor Hugo',      5, 'Auxiliar Financeiro',     48000, '2023-09-10', 'victor@emp.com',    'São Paulo',      1),
            # Suporte
            (23, 'Wanderlei Gomes',  6, 'Gerente de Suporte',      98000, '2018-02-28', 'wanderlei@emp.com', 'Curitiba',       1),
            (24, 'Ximena Borges',    6, 'Analista de Suporte N2',  65000, '2020-07-15', 'ximena@emp.com',    'Curitiba',       1),
            (25, 'Yago Ribeiro',     6, 'Analista de Suporte N1',  52000, '2022-05-03', 'yago@emp.com',      'Curitiba',       1),
            (26, 'Zara Monteiro',    6, 'Especialista em QA',      70000, '2021-01-19', 'zara@emp.com',      'Florianópolis',  1),
            # Inativos
            (27, 'André Lopes',      1, 'Engenheiro Sênior',       92000, '2019-06-10', 'andre@emp.com',     'São Paulo',      0),
            (28, 'Beatriz Campos',   2, 'Vendedora Pleno',         68000, '2020-04-25', 'beatriz@emp.com',   'Rio de Janeiro', 0),
        ]

        projetos = [
            (1,  'Sistema ERP v3',           1, 250000, 'Em Andamento', '2024-01-10', '2024-12-31', 'Alta'),
            (2,  'App Mobile de Vendas',      2, 120000, 'Em Andamento', '2024-03-01', '2024-09-30', 'Alta'),
            (3,  'Portal RH Digital',         3,  80000, 'Concluído',    '2023-06-01', '2024-02-28', 'Média'),
            (4,  'Campanha Verão 2024',        4,  60000, 'Concluído',    '2023-11-01', '2024-02-29', 'Média'),
            (5,  'Automação Financeira',       5,  90000, 'Em Andamento', '2024-02-15', '2024-11-30', 'Alta'),
            (6,  'Central de Atendimento IA', 6, 110000, 'Em Andamento', '2024-04-01', '2025-03-31', 'Alta'),
            (7,  'Migração para Cloud',        1, 200000, 'Planejamento', '2024-07-01', '2025-06-30', 'Média'),
            (8,  'Integração CRM',             2,  75000, 'Em Andamento', '2024-05-01', '2024-10-31', 'Média'),
            (9,  'Treinamento Corporativo',    3,  40000, 'Concluído',    '2023-09-01', '2024-01-31', 'Baixa'),
            (10, 'Rebranding 2024',            4,  95000, 'Planejamento', '2024-08-01', '2025-01-31', 'Alta'),
        ]

        alocacoes = [
            (1,  1,  1, 20, 'Tech Lead'),    (2,  2,  1, 30, 'Desenvolvedor'),
            (3,  3,  1, 25, 'Desenvolvedora'),(4,  5,  1, 15, 'DevOps'),
            (5,  1,  7, 10, 'Arquiteto'),    (6,  6,  2, 10, 'Sponsor'),
            (7,  7,  2, 20, 'Vendedora'),    (8,  10, 2, 25, 'Analista'),
            (9,  11, 3, 10, 'Sponsor'),      (10, 12, 3, 20, 'Analista'),
            (11, 19, 5, 15, 'Sponsor'),      (12, 20, 5, 30, 'Contador'),
            (13, 23, 6, 15, 'Gerente'),      (14, 26, 6, 30, 'QA Lead'),
        ]

        avaliacoes = [
            (1, 1,  2023, 9.5, 'Liderança excepcional e entrega consistente'),
            (2, 2,  2023, 8.8, 'Forte desempenho técnico e colaboração'),
            (3, 3,  2023, 8.2, 'Boa evolução ao longo do ano'),
            (4, 4,  2023, 7.5, 'Curva de aprendizado esperada para júnior'),
            (5, 6,  2023, 9.2, 'Superou metas de vendas em 30%'),
            (6, 7,  2023, 8.5, 'Excelente relacionamento com clientes'),
            (7, 19, 2023, 9.0, 'Controle financeiro exemplar'),
            (8, 23, 2023, 8.7, 'Equipe bem gerenciada, baixo turnover'),
            (9, 1,  2024, 9.7, 'Entrega do ERP dentro do prazo e orçamento'),
            (10,6,  2024, 8.9, 'Maior volume de contratos fechados da história'),
        ]

        c.executemany('INSERT OR REPLACE INTO departamentos VALUES (?,?,?,?,?)', departamentos)
        c.executemany('INSERT OR REPLACE INTO funcionarios  VALUES (?,?,?,?,?,?,?,?,?)', funcionarios)
        c.executemany('INSERT OR REPLACE INTO projetos      VALUES (?,?,?,?,?,?,?,?)', projetos)
        c.executemany('INSERT OR REPLACE INTO alocacoes     VALUES (?,?,?,?,?)', alocacoes)
        c.executemany('INSERT OR REPLACE INTO avaliacoes    VALUES (?,?,?,?,?)', avaliacoes)

        print(f'Banco "{caminho}" configurado:')
        for t in ['departamentos', 'funcionarios', 'projetos', 'alocacoes', 'avaliacoes']:
            n = c.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
            print(f'  • {t}: {n} registros')


def obter_esquema_banco(conexao):
    """Lê a estrutura real do banco em tempo de execução (RAG dinâmico)."""
    c = conexao.cursor()
    c.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    tabelas = [r[0] for r in c.fetchall()]
    linhas = []
    for t in tabelas:
        c.execute(f"PRAGMA table_info({t});")
        colunas = ', '.join(f"{r[1]} ({r[2]})" for r in c.fetchall())
        linhas.append(f"Tabela '{t}': {colunas}")
    return "\n".join(linhas)


configurar_banco_dados()

Banco "empresa.db" configurado:
  • departamentos: 6 registros
  • funcionarios: 28 registros
  • projetos: 10 registros
  • alocacoes: 14 registros
  • avaliacoes: 10 registros


In [3]:
from datetime import datetime
from collections import deque

# ──────────────────────────────────────────────
# PERCEPÇÃO DE TEMPO
# ──────────────────────────────────────────────

def obter_contexto_temporal() -> str:
    agora = datetime.now()
    DIAS   = ['Segunda-feira','Terça-feira','Quarta-feira',
              'Quinta-feira','Sexta-feira','Sábado','Domingo']
    MESES  = ['janeiro','fevereiro','março','abril','maio','junho',
              'julho','agosto','setembro','outubro','novembro','dezembro']
    return (f"{DIAS[agora.weekday()]}, {agora.day} de "
            f"{MESES[agora.month-1]} de {agora.year} — {agora.strftime('%H:%M')}")


# ──────────────────────────────────────────────
# MEMÓRIA DE CURTO E LONGO PRAZO
# ──────────────────────────────────────────────
class SistemaMemoria:
    """
    Curto prazo  → buffer circular com as últimas N trocas (contexto imediato).
    Longo prazo  → dicionário de fatos extraídos por palavras-chave temáticas.
    """
    _TEMAS = ['funcionário', 'departamento', 'salário', 'projeto',
              'orçamento', 'avaliação', 'cargo', 'alocação']

    def __init__(self, max_curto_prazo: int = 6):
        self._curto:  deque  = deque(maxlen=max_curto_prazo)
        self._longo:  dict   = {}
        self.total_trocas: int = 0

    def registrar(self, pergunta: str, resposta: str) -> None:
        self._curto.append({'u': pergunta, 'a': resposta})
        self.total_trocas += 1

        for tema in self._TEMAS:
            if tema in pergunta.lower():
                self._longo[f"ultima_{tema}"] = pergunta[:120]

    def contexto_curto(self) -> str:
        if not self._curto:
            return "Sem histórico nessa sessão."
        return "\n".join(
            f"Usuário: {t['u']}\nAgente: {t['a'][:100]}…"
            for t in self._curto
        )

    def contexto_longo(self) -> str:
        if not self._longo:
            return ""
        return "Tópicos abordados anteriormente:\n" + "\n".join(
            f"  • {k}: {v}" for k, v in self._longo.items()
        )

    def resumo_completo(self) -> str:
        partes = [self.contexto_longo(), self.contexto_curto()]
        return "\n".join(p for p in partes if p)


# ──────────────────────────────────────────────
# BUSCA NA WEB (DuckDuckGo, sem chave de API)
# ──────────────────────────────────────────────
def buscar_na_web(query: str, max_resultados: int = 4) -> str:
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            resultados = list(ddgs.text(query, max_results=max_resultados))
        if not resultados:
            return "Nenhum resultado encontrado pra essa busca."
        linhas = [f"• {r.get('title','')}: {r.get('body','')[:220]}"
                  for r in resultados]
        return "\n".join(linhas)
    except Exception as e:
        return f"Busca indisponível no momento: {e}"


print("Ferramentas carregadas: percepção temporal ✓  memória ✓  busca web ✓")

Ferramentas carregadas: percepção temporal ✓  memória ✓  busca web ✓


In [4]:
import dspy

# configs do modelo local ──────────────────────────────────────────────

lm = dspy.LM('ollama/gemma3:1b', api_base='http://localhost:11434')
dspy.settings.configure(lm=lm)


# ASSINATURAS ───────────────────────────────────────────────────────────────

class ClassificadorIntencao(dspy.Signature):
    """Classifique a mensagem do usuário em exatamente uma categoria:
    • 'BANCO_DADOS' — pergunta sobre funcionários, salários, departamentos,
      projetos, orçamentos, alocações ou avaliações da empresa.
    • 'BUSCA_WEB'   — precisa de informações atuais da internet, notícias,
      cotações, clima ou qualquer dado externo à empresa.
    • 'GERAL'       — cumprimentos, perguntas filosóficas, conversas livres
      ou qualquer outro assunto.
    Responda APENAS com uma dessas três palavras, sem pontuação."""
    contexto_temporal    = dspy.InputField(desc="Data e hora atuais")
    historico_conversa   = dspy.InputField(desc="Últimas mensagens da sessão")
    pergunta_usuario     = dspy.InputField(desc="Mensagem atual do usuário")
    intencao             = dspy.OutputField(desc="BANCO_DADOS | BUSCA_WEB | GERAL")


class RespostaGeral(dspy.Signature):
    """Você é um assistente corporativo inteligente. Responda de forma natural,
    usando o horário e o histórico como contexto quando for relevante.
    Lembre gentilmente que seu foco principal é análise de dados da empresa."""
    contexto_temporal  = dspy.InputField()
    historico_conversa = dspy.InputField()
    pergunta_usuario   = dspy.InputField()
    resposta           = dspy.OutputField(desc="Resposta amigável em português")


class SinteseBuscaWeb(dspy.Signature):
    """Sintetize os resultados de busca respondendo à pergunta do usuário
    de forma clara, objetiva e em português. Cite as fontes quando relevante."""
    contexto_temporal  = dspy.InputField()
    pergunta_usuario   = dspy.InputField()
    resultados_busca   = dspy.InputField(desc="Trechos retornados pela busca")
    resposta_sintetizada = dspy.OutputField(desc="Síntese em português")


class GeracaoSQL(dspy.Signature):
    """Você é especialista em SQLite. Converta a pergunta em SQL válido
    usando EXATAMENTE as tabelas e colunas do esquema fornecido.
    Use o histórico para resolver referências a consultas anteriores.
    Retorne APENAS o código SQL — sem markdown, sem explicações."""
    esquema_banco      = dspy.InputField(desc="Esquema completo do banco")
    contexto_temporal  = dspy.InputField(desc="Data atual para filtros temporais")
    historico_conversa = dspy.InputField(desc="Contexto para resolver referências")
    pergunta_usuario   = dspy.InputField()
    codigo_sql         = dspy.OutputField(desc="SQL válido e executável, sem blocos de código")


class RefinamentoSQL(dspy.Signature):
    """O SQL gerado falhou. Analise o erro retornado e produza uma versão
    corrigida. Retorne APENAS o SQL corrigido, sem explicações."""
    esquema_banco    = dspy.InputField()
    pergunta_usuario = dspy.InputField()
    sql_com_erro     = dspy.InputField(desc="SQL que gerou o erro")
    mensagem_erro    = dspy.InputField(desc="Mensagem de erro do SQLite")
    sql_corrigido    = dspy.OutputField(desc="SQL corrigido e válido")


# MÓDULO DO AGENTE ──────────────────────────────────────────────────────────

class AgenteBancoDadosProfissional(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classificador  = dspy.Predict(ClassificadorIntencao)
        self.chat_geral     = dspy.ChainOfThought(RespostaGeral)
        self.sintetizador   = dspy.ChainOfThought(SinteseBuscaWeb)
        self.gerador_sql    = dspy.ChainOfThought(GeracaoSQL)
        self.refinador_sql  = dspy.ChainOfThought(RefinamentoSQL)

    @staticmethod
    def _limpar_sql(texto: str) -> str:
        return texto.strip().replace('```sql', '').replace('```', '').strip()

    @staticmethod
    def _normalizar_intencao(texto: str) -> str:
        t = texto.strip().upper()
        if any(k in t for k in ('BANCO', 'SQL', 'DADOS', 'DB')):
            return 'BANCO_DADOS'
        if any(k in t for k in ('WEB', 'BUSCA', 'INTERNET', 'NEWS')):
            return 'BUSCA_WEB'
        return 'GERAL'

    def forward(self, pergunta: str, conexao=None, memoria: SistemaMemoria = None):
        tempo    = obter_contexto_temporal()
        historico = memoria.resumo_completo() if memoria else "Sem histórico."

        # 1. Roteamento
        clf      = self.classificador(contexto_temporal=tempo,
                                      historico_conversa=historico,
                                      pergunta_usuario=pergunta)
        intencao = self._normalizar_intencao(clf.intencao)

        # 2. Bate-papo geral
        if intencao == 'GERAL':
            r = self.chat_geral(contexto_temporal=tempo,
                                historico_conversa=historico,
                                pergunta_usuario=pergunta)
            if memoria:
                memoria.registrar(pergunta, r.resposta)
            return dspy.Prediction(tipo='chat', intencao=intencao,
                                   resposta=r.resposta)

        # 3. Busca na web
        if intencao == 'BUSCA_WEB':
            raw = buscar_na_web(pergunta)
            r   = self.sintetizador(contexto_temporal=tempo,
                                    pergunta_usuario=pergunta,
                                    resultados_busca=raw)
            if memoria:
                memoria.registrar(pergunta, r.resposta_sintetizada)
            return dspy.Prediction(tipo='web', intencao=intencao,
                                   resposta=r.resposta_sintetizada,
                                   fontes=raw)

        # 4. Consulta ao banco de dados
        esquema = obter_esquema_banco(conexao)
        gen     = self.gerador_sql(esquema_banco=esquema,
                                   contexto_temporal=tempo,
                                   historico_conversa=historico,
                                   pergunta_usuario=pergunta)
        sql = self._limpar_sql(gen.codigo_sql)

        try:
            cur  = conexao.execute(sql)
            rows = cur.fetchall()
            cols = [d[0] for d in cur.description] if cur.description else []
            if memoria:
                memoria.registrar(pergunta, f"SQL executado — {len(rows)} linha(s)")
            return dspy.Prediction(tipo='sql', intencao=intencao,
                                   sql=sql, resultados=rows,
                                   colunas=cols, erro=None)

        except Exception as e1:
            # 4a. Auto-correção
            ref = self.refinador_sql(esquema_banco=esquema,
                                     pergunta_usuario=pergunta,
                                     sql_com_erro=sql,
                                     mensagem_erro=str(e1))
            sql2 = self._limpar_sql(ref.sql_corrigido)

            try:
                cur  = conexao.execute(sql2)
                rows = cur.fetchall()
                cols = [d[0] for d in cur.description] if cur.description else []
                if memoria:
                    memoria.registrar(pergunta, f"SQL corrigido — {len(rows)} linha(s)")
                return dspy.Prediction(tipo='sql', intencao=intencao,
                                       sql=sql2, resultados=rows,
                                       colunas=cols, erro=None)
            except Exception as e2:
                return dspy.Prediction(tipo='sql', intencao=intencao,
                                       sql=sql2, resultados=None,
                                       colunas=[], erro=str(e2))


print("Agente configurado com sucesso!")

Agente configurado com sucesso!


In [5]:
import sqlite3
from dspy.teleprompt import BootstrapFewShot

_conn_treino = sqlite3.connect('empresa.db')

# Dataset
exemplos_treino = [
    # ---------------------------------------------------------
    # 📊 INTENÇÃO: BANCO DE DADOS
    # ---------------------------------------------------------

    # filtros simples e contagens
    dspy.Example(pergunta="Quantos funcionários ativos existem na empresa?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Qual é o orçamento total do departamento de Engenharia?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Liste os três funcionários com os maiores salários.", conexao=_conn_treino).with_inputs("pergunta", "conexao"),

    # múltiplas condições e datas
    dspy.Example(pergunta="Quais projetos estão 'Em Andamento' e têm prioridade 'Alta'?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Mostre as avaliações do ano de 2023 com notas acima de 9.0.", conexao=_conn_treino).with_inputs("pergunta", "conexao"),

    # agregações (GROUP BY)
    dspy.Example(pergunta="Qual é a média salarial agrupada por departamento?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),

    # relacionamentos (JOINs implícitos/explícitos)
    dspy.Example(pergunta="Quem são os gerentes dos departamentos de Vendas e Marketing?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Quantas horas semanais a Alice Silva está alocada em projetos?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),

    # ---------------------------------------------------------
    # 💬 INTENÇÃO: BATE-PAPO GERAL
    # ---------------------------------------------------------
    # cumprimentos e identidade
    dspy.Example(pergunta="Olá, bom dia! Tudo bem com você?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Quem é você e o que você faz?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),

    # testes de limite (pra ele não tentar buscar isso no banco)
    dspy.Example(pergunta="Pode me contar uma piada curta sobre programação?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Quero apenas conversar um pouco para testar o sistema.", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Escreva um pequeno poema sobre a rotina corporativa.", conexao=_conn_treino).with_inputs("pergunta", "conexao"),

    # ---------------------------------------------------------
    # 🌐 INTENÇÃO: BUSCA NA WEB
    # ---------------------------------------------------------
    # clima e atualidades
    dspy.Example(pergunta="Qual é a previsão do tempo para São José dos Campos hoje?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Quais são as principais notícias de tecnologia do dia?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),

    # finanças e dados factuais externos
    dspy.Example(pergunta="Qual foi o valor de fechamento do dólar comercial ontem?", conexao=_conn_treino).with_inputs("pergunta", "conexao"),
    dspy.Example(pergunta="Quais são as novidades sobre o lançamento de novos modelos de inteligência artificial?", conexao=_conn_treino).with_inputs("pergunta", "conexao")
]

def metrica_resposta_valida(exemplo, pred, trace=None):
    """Verifica se o agente entregou uma resposta coerente."""
    tipo = getattr(pred, 'tipo', None)
    if tipo == 'sql':
        sql  = getattr(pred, 'sql', '') or ''
        rows = getattr(pred, 'resultados', None)
        erro = getattr(pred, 'erro', None)
        sql_ok = len(sql) > 5 and sql.strip().upper().startswith(
            ('SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE'))
        return sql_ok and erro is None
    if tipo in ('chat', 'web'):
        return len(getattr(pred, 'resposta', '') or '') > 10
    return False

print("Compilando agente com BootstrapFewShot...")
agente_base      = AgenteBancoDadosProfissional()
otimizador       = BootstrapFewShot(
    metric=metrica_resposta_valida,
    max_bootstrapped_demos=3,
    max_labeled_demos=len(exemplos_treino)
)
agente_otimizado = otimizador.compile(agente_base, trainset=exemplos_treino)
_conn_treino.close()
print("Agente otimizado e pronto pra uso!")

Compilando agente com BootstrapFewShot...


 24%|██▎       | 4/17 [00:32<01:44,  8.04s/it]

Bootstrapped 3 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Agente otimizado e pronto pra uso!


In [6]:
import sqlite3

COMANDOS_ESPECIAIS = {
    'memoria' : 'exibe o histórico da sessão',
    'schema'  : 'mostra o esquema atual do banco',
    'tempo'   : 'exibe a data e hora atual',
    'ajuda'   : 'lista os comandos disponíveis',
    'sair'    : 'encerra o assistente',
}

def _linha(char='─', n=62):
    print(char * n)

def iniciar_chat():
    agente  = agente_otimizado
    memoria = SistemaMemoria(max_curto_prazo=6)

    _linha('═')
    print("  Assistente de Dados Corporativos")
    print(f"  {obter_contexto_temporal()}")
    print(f"  Comandos: {' | '.join(COMANDOS_ESPECIAIS)}")
    _linha('═')
    print()

    with sqlite3.connect('empresa.db') as conexao:
        while True:
            try:
                pergunta = input("Você: ").strip()
            except (EOFError, KeyboardInterrupt):
                print("\nEncerrando o assistente.")
                break

            if not pergunta:
                continue

            cmd = pergunta.lower()

            # comandos especiais
            if cmd in ('sair', 'exit', 'quit'):
                print(f"Até logo! ({memoria.total_trocas} interações nesta sessão)")
                break

            if cmd == 'ajuda':
                print("\nComandos disponíveis:")
                for c, desc in COMANDOS_ESPECIAIS.items():
                    print(f"  {c:<10} {desc}")
                print()
                continue

            if cmd == 'memoria':
                print("\n" + memoria.resumo_completo() or "Nenhum histórico ainda.")
                print()
                continue

            if cmd == 'schema':
                print("\nEsquema atual do banco:")
                print(obter_esquema_banco(conexao))
                print()
                continue

            if cmd == 'tempo':
                print(f"\n{obter_contexto_temporal()}\n")
                continue

            # inferência
            print("Pensando…\n")
            pred     = agente(pergunta=pergunta, conexao=conexao, memoria=memoria)
            tipo     = getattr(pred, 'tipo', 'sql')
            intencao = getattr(pred, 'intencao', '?')

            print(f"[{intencao}]")

            if tipo == 'chat':
                print(f"Assistente: {pred.resposta}")

            elif tipo == 'web':
                print(f"Assistente: {pred.resposta}")

            else:  # sql
                print(f"SQL gerado:\n  {pred.sql}\n")
                erro = getattr(pred, 'erro', None)
                if erro:
                    print(f"Não foi possível executar a consulta: {erro}")
                else:
                    rows = pred.resultados or []
                    cols = getattr(pred, 'colunas', [])
                    if rows:
                        header = f"({len(rows)} linha{'s' if len(rows)>1 else ''})"
                        if cols:
                            header += f" — {', '.join(cols)}"
                        print(f"Resultados {header}:")
                        for row in rows:
                            valores = ' | '.join(str(v) for v in row)
                            print(f"  • {valores}")
                    else:
                        print("Nenhum registro encontrado.")

            print()
            _linha()
            print()

iniciar_chat()

══════════════════════════════════════════════════════════════
  Assistente de Dados Corporativos
  Quinta-feira, 7 de maio de 2026 — 00:12
  Comandos: memoria | schema | tempo | ajuda | sair
══════════════════════════════════════════════════════════════

Você: olá boa noite
Pensando…

[GERAL]
Assistente: Boa noite! Espero que tenha um ótimo dia.

──────────────────────────────────────────────────────────────

Você: igualmente, que horas são agora? pelo horario de brasilia
Pensando…

[GERAL]
Assistente: A hora atual é 00:13 da manhã, Brasília.

──────────────────────────────────────────────────────────────

Você: na verdade, são 21:13. de onde voce achou esses dados (que são 00:13)?
Pensando…



/tmp/ipykernel_12608/3647058003.py:68: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use ti

[BUSCA_WEB]
Assistente: A busca não encontrou nenhuma informação relacionada à pergunta "quais são os dados de 00:13?".

──────────────────────────────────────────────────────────────



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


Encerrando o assistente.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag